# Project 24 — A* Search: Real-World London Route Planner

**Business problem:** find an efficient real road route between two locations while exploring far fewer irrelevant nodes than an uninformed search.

This project downloads a **real OpenStreetMap driving network around central London** with OSMnx, implements A* from scratch, uses geographic straight-line distance as an admissible heuristic, reconstructs the route, compares A* with Dijkstra and visualises both routes.

**Big-company connection:** this maps directly to the graph-routing problem solved by navigation and mobility products such as **Google Maps and Uber**. Their production systems are vastly more sophisticated, but A* demonstrates the core idea: combine distance travelled with an estimate of distance remaining.

For a consistent heuristic, both edge costs and the Haversine heuristic are measured in metres.

In [ ]:
# Colab setup
%pip -q install osmnx

import heapq, math, time
import numpy as np
import pandas as pd
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt

ox.settings.use_cache = True
ox.settings.log_console = False

# Central London network containing both endpoints
centre = (51.5019, -0.1330)
G = ox.graph_from_point(centre, dist=4500, network_type="drive", simplify=True)

# Buckingham Palace -> Big Ben
start_latlon = (51.5014, -0.1419)
goal_latlon = (51.5007, -0.1246)

start = ox.distance.nearest_nodes(G, X=start_latlon[1], Y=start_latlon[0])
goal = ox.distance.nearest_nodes(G, X=goal_latlon[1], Y=goal_latlon[0])

print(f"Road graph nodes: {G.number_of_nodes():,}")
print(f"Road graph edges: {G.number_of_edges():,}")
print("Start node:", start)
print("Goal node:", goal)

## A* implementation

The priority is:

**f(n) = g(n) + h(n)**

- **g(n):** road distance already travelled.
- **h(n):** straight-line Haversine distance to the destination.
- If the heuristic never overestimates remaining road distance, A* remains optimal.

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2-lat1)
    dl = math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

def heuristic(graph, node, goal):
    n = graph.nodes[node]
    g = graph.nodes[goal]
    return haversine_m(n["y"], n["x"], g["y"], g["x"])

def edge_length(graph, u, v):
    data = graph.get_edge_data(u, v)
    return min(attrs.get("length", 1.0) for attrs in data.values())

def astar_search(graph, start, goal):
    pq = [(heuristic(graph, start, goal), 0.0, start)]
    g_score = {start: 0.0}
    parent = {start: None}
    closed = set()
    expanded = 0

    while pq:
        _, current_g, node = heapq.heappop(pq)

        if node in closed:
            continue
        closed.add(node)
        expanded += 1

        if node == goal:
            path = []
            cur = node
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            return path[::-1], current_g, expanded

        for nbr in graph.successors(node):
            tentative = current_g + edge_length(graph, node, nbr)
            if tentative < g_score.get(nbr, float("inf")):
                g_score[nbr] = tentative
                parent[nbr] = node
                f_score = tentative + heuristic(graph, nbr, goal)
                heapq.heappush(pq, (f_score, tentative, nbr))

    return None, float("inf"), expanded

## Real route example — Buckingham Palace to Big Ben

In [ ]:
t0 = time.perf_counter()
a_path, a_dist, a_expanded = astar_search(G, start, goal)
a_ms = (time.perf_counter()-t0)*1000

t0 = time.perf_counter()
d_path = nx.shortest_path(G, start, goal, weight="length", method="dijkstra")
d_ms = (time.perf_counter()-t0)*1000
d_dist = sum(edge_length(G, u, v) for u, v in zip(d_path[:-1], d_path[1:]))

print("A* route distance:", round(a_dist/1000, 3), "km")
print("Dijkstra route distance:", round(d_dist/1000, 3), "km")
print("Distance difference:", round(a_dist-d_dist, 3), "m")
print("A* nodes expanded:", a_expanded)
print("A* runtime:", round(a_ms, 3), "ms")
print("NetworkX Dijkstra runtime:", round(d_ms, 3), "ms")

In [ ]:
# Visualise the A* route
fig, ax = ox.plot_graph_route(
    G,
    a_path,
    route_linewidth=4,
    node_size=0,
    bgcolor="white",
    show=False,
    close=False
)
ax.set_title("A* route: Buckingham Palace → Big Ben")
plt.show()

## Compare A* against Dijkstra explicitly

Dijkstra is A* with **h(n)=0**. The comparison below counts node expansions for both custom implementations, which is more informative than runtime alone on a single query.

In [ ]:
def dijkstra_search(graph, start, goal):
    pq = [(0.0, start)]
    dist = {start: 0.0}
    parent = {start: None}
    closed = set()
    expanded = 0

    while pq:
        current_g, node = heapq.heappop(pq)
        if node in closed:
            continue
        closed.add(node)
        expanded += 1

        if node == goal:
            path = []
            cur = node
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            return path[::-1], current_g, expanded

        for nbr in graph.successors(node):
            tentative = current_g + edge_length(graph, node, nbr)
            if tentative < dist.get(nbr, float("inf")):
                dist[nbr] = tentative
                parent[nbr] = node
                heapq.heappush(pq, (tentative, nbr))

    return None, float("inf"), expanded

d2_path, d2_dist, d_expanded = dijkstra_search(G, start, goal)

comparison = pd.DataFrame([
    {"algorithm":"A*", "distance_km":a_dist/1000, "nodes_expanded":a_expanded},
    {"algorithm":"Dijkstra", "distance_km":d2_dist/1000, "nodes_expanded":d_expanded},
])
display(comparison)

plt.figure(figsize=(7,4))
plt.bar(comparison["algorithm"], comparison["nodes_expanded"])
plt.ylabel("Nodes expanded")
plt.title("Search efficiency on the same real London route")
plt.show()

## Why A* is different from BFS and DFS

| Algorithm | Best use | Guarantees |
|---|---|---|
| BFS | Unweighted shortest path | Minimum number of edges |
| DFS | Deep/exhaustive traversal | Reachability, not shortest path |
| Dijkstra | Weighted shortest path | Optimal if weights are non-negative |
| A* | Weighted route with heuristic | Optimal with an admissible/consistent heuristic |

### Production extensions
A real navigation company would add live traffic, turn restrictions, road speeds, incidents, tolls, vehicle constraints, map matching, contraction hierarchies or other routing accelerators, distributed serving infrastructure and continuous quality monitoring.

**CV bullet:** Built a real-world A* road-routing engine on OpenStreetMap London data, implementing Haversine-guided graph search, path reconstruction, Dijkstra benchmarking, route visualisation and search-efficiency comparison for a navigation-style use case.